# Notebook 03 — Tái hiện và đánh giá baseline B1

Notebook chỉ nhận run `b1_ctc_v2` đã hoàn thành đủ 20 epoch. `checkpoint_best.pt` được chọn bằng
validation WER (tie-break CER, rồi epoch sớm hơn). Audio test được đọc trực tiếp từ ZIP shard
của Notebook 01; thiếu một mẫu sẽ dừng. Output báo corpus WER/CER tổng và theo ba miền.


In [ ]:
# 1. Nạp source và cài dependencies
%cd /kaggle/working
%env HF_HOME=/kaggle/working/.cache/huggingface
%env HF_DATASETS_CACHE=/kaggle/working/.cache/huggingface/datasets
%env TRANSFORMERS_CACHE=/kaggle/working/.cache/huggingface/hub
%env TORCH_HOME=/kaggle/working/.cache/torch

import hashlib
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

work_dir = Path("/kaggle/working/vi_asr_project")
preferred_zip = Path("/kaggle/input/vi-asr-code/vi_asr_project.zip")
preferred_project = Path("/kaggle/input/vi-asr-code/vi_asr_project")
zip_candidates = sorted(Path("/kaggle/input").rglob("vi_asr_project.zip"))
project_candidates = sorted(
    path.parent for path in Path("/kaggle/input").rglob("vi_asr_project/pyproject.toml")
)
dataset_projects = [path for path in project_candidates if "vi-asr-code" in path.parts]
dataset_zips = [path for path in zip_candidates if "vi-asr-code" in path.parts]
handoff_projects = [
    path for path in project_candidates
    if (path.parent / "artifacts" / "vi_asr_handoff.json").is_file()
]

source_path = None
source_kind = None
if (preferred_project / "pyproject.toml").is_file():
    # Kaggle thường bung ZIP đã upload thành thư mục read-only này.
    source_path, source_kind = preferred_project, "directory"
elif len(dataset_projects) == 1:
    # Kaggle image mới mount theo /kaggle/input/datasets/<owner>/<slug>/...
    source_path, source_kind = dataset_projects[0], "directory"
elif len(dataset_projects) > 1:
    raise RuntimeError(f"Có nhiều thư mục source trong dataset vi-asr-code: {dataset_projects}")
elif preferred_zip.is_file():
    source_path, source_kind = preferred_zip, "zip"
elif len(dataset_zips) == 1:
    source_path, source_kind = dataset_zips[0], "zip"
elif len(dataset_zips) > 1:
    raise RuntimeError(f"Có nhiều source ZIP trong dataset vi-asr-code: {dataset_zips}")
elif len(zip_candidates) == 1:
    source_path, source_kind = zip_candidates[0], "zip"
elif len(zip_candidates) > 1:
    raise RuntimeError(f"Có nhiều source ZIP, chỉ giữ một bản: {zip_candidates}")
elif len(handoff_projects) == 1:
    source_path, source_kind = handoff_projects[0], "directory"
elif not handoff_projects and len(project_candidates) == 1:
    source_path, source_kind = project_candidates[0], "directory"
else:
    raise FileNotFoundError(
        "Không xác định được duy nhất source project. Hãy attach dataset vi-asr-code "
        "hoặc output Notebook 01."
    )

if not (work_dir / "pyproject.toml").is_file():
    if source_kind == "zip":
        with zipfile.ZipFile(source_path, "r") as archive:
            archive.extractall("/kaggle/working")
        print("Đã giải nén source:", source_path)
    else:
        shutil.copytree(source_path, work_dir)
        print("Đã copy source:", source_path)

def sha256_source(path):
    path = Path(path)
    digest = hashlib.sha256()
    files = [path] if path.is_file() else sorted(item for item in path.rglob("*") if item.is_file())
    for item in files:
        relative = item.name if path.is_file() else item.relative_to(path).as_posix()
        digest.update(relative.encode("utf-8"))
        digest.update(b"\0")
        with item.open("rb") as stream:
            for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
                digest.update(block)
    return digest.hexdigest()

source_sha256 = sha256_source(source_path)
print({"source": str(source_path), "kind": source_kind, "sha256": source_sha256})

os.environ["PYTHONPATH"] = str(work_dir / "src")
if str(work_dir / "src") not in sys.path:
    sys.path.insert(0, str(work_dir / "src"))

%cd /kaggle/working/vi_asr_project
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.in"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import vi_asr
print("vi_asr:", vi_asr.__file__)


In [ ]:
# 2. Kích hoạt artifact NB1 — không giải nén audio vào /kaggle/working
import subprocess
import sys

subprocess.run([
    sys.executable, "scripts/kaggle_handoff.py", "restore",
    "--input-root", "/kaggle/input",
    "--working-root", "/kaggle/working",
    "--runtime-paths", "configs/paths/kaggle_runtime.yaml",
], check=True)


In [ ]:
# 3. Xác minh run B1 đã hoàn thành đủ fixed budget 20 epoch
import hashlib
import json
from pathlib import Path
import torch

def find_one(filename):
    candidates = sorted(
        path for path in Path("/kaggle/input").rglob(filename)
        if path.parent.name == "b1_ctc_v2"
    )
    if len(candidates) != 1:
        raise RuntimeError(
            f"Cần đúng 1 {filename} cho b1_ctc_v2, tìm thấy {len(candidates)}: {candidates}"
        )
    return candidates[0]

BEST_CHECKPOINT = find_one("checkpoint_best.pt")
LATEST_CHECKPOINT = find_one("checkpoint_latest.pt")
latest = torch.load(LATEST_CHECKPOINT, map_location="cpu", weights_only=False)
best = torch.load(BEST_CHECKPOINT, map_location="cpu", weights_only=False)
assert int(latest["epoch"]) == 20, f"B1 mới chạy đến epoch {latest['epoch']}/20"
assert latest["config"]["experiment"]["name"] == "b1_ctc"
assert latest["config"]["experiment"]["protocol_version"] == "b1_repro_v2"
assert int(latest["config"]["experiment"]["seed"]) == 42
assert best["metrics"]["best_epoch"] == best["epoch"]
print({
    "latest_epoch": latest["epoch"],
    "best_epoch": best["epoch"],
    "best_valid_wer": best["metrics"]["best_wer"],
    "best_valid_cer": best["metrics"]["best_cer"],
    "best_checkpoint": str(BEST_CHECKPOINT),
})


In [ ]:
# 4. Decode frozen test split đúng một lần bằng best validation checkpoint
subprocess.run([
    sys.executable, "-m", "vi_asr", "evaluate-asr",
    "--config", "configs/base.yaml",
    "--paths", "configs/paths/kaggle_runtime.yaml",
    "--checkpoint", str(BEST_CHECKPOINT),
    "--split", "test",
    "--run-dir", "/kaggle/working/eval/asr_b1",
], check=True)


In [ ]:
# 5. Gate cuối: coverage 100%, provenance và báo cáo tái hiện B1
eval_path = Path("/kaggle/working/eval/asr_b1/eval_test.json")
result = json.loads(eval_path.read_text(encoding="utf-8"))
assert result["coverage"] == {"decoded": 2026, "manifest": 2026}
assert result["checkpoint_epoch"] == best["epoch"]
assert result["checkpoint_selection_metrics"]["best_epoch"] == best["epoch"]
assert set(result["metrics_by_region"]) == {"north", "central", "south"}

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

report = {
    "status": "completed",
    "baseline": "B1: ViP-VL + Linear CTC",
    "seed": 42,
    "fixed_budget_epochs": 20,
    "selected_epoch": best["epoch"],
    "selection_metric": "validation WER; tie-break CER then earliest epoch",
    "test_samples": result["coverage"]["decoded"],
    "test_metrics": result["metrics"],
    "metrics_by_region": result["metrics_by_region"],
    "checkpoint_sha256": result["checkpoint_sha256"],
    "training_config_sha256": result["training_config_sha256"],
    "source_sha256": source_sha256,
}
report_path = Path("/kaggle/working/artifacts/b1_reproduction_report.json")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(report, indent=2, ensure_ascii=False))
print("PASS B1 REPRODUCTION:", report_path)
